# Código de Extracción

## Librerías

In [1]:
# --- Librerías estándar ---
import os
import json
import re
import time
import copy
import shutil
import logging
from itertools import permutations
from html import unescape
from glob import glob
import pandas as pd
import numpy as np
from tqdm import tqdm

# --- IA ---
from dotenv import load_dotenv
import openai
from openai import OpenAI
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

# --- Pydantic ---
from enum import Enum
from typing import List, Optional, Literal, Dict, Union, Annotated, Any, Tuple
from pydantic import BaseModel, Field, ValidationError, field_validator, ConfigDict, constr, conint, conlist, model_validator, create_model
from pydantic.types import StringConstraints

## Paths y API Keys

In [2]:
path = os.path.join(os.environ['USERPROFILE'], "RUTA")

# Cargar API KEY
load_dotenv(dotenv_path=os.path.join(path, "Códigos", ".env"))
api_key_openai = os.getenv("OPENAI_API_KEY")
client = openai.OpenAI(api_key=api_key_openai)

## Carga de Datos

### Listado Conservadores

In [3]:
df_cbr = pd.read_excel(os.path.join(path, "Archivos Auxiliares", "Listados Auxiliares.xlsx"), sheet_name="Listado CBRs")
Region = Enum("Region", {row["region"]: row["nombre_region"] for _, row in df_cbr.drop_duplicates("region").iterrows()}, type=str)
Conservador = Enum("Conservador", {row["conservador"]: row["nombre_conservador"] for _, row in df_cbr.drop_duplicates("conservador").iterrows()}, type=str)
region_por_conservador = {row["nombre_conservador"]: row["nombre_region"] for _, row in df_cbr.iterrows()}

### Listado Tipos de Transacción

In [4]:
df_config = pd.read_excel(os.path.join(path, "Archivos Auxiliares", "Listados Auxiliares.xlsx"), sheet_name="Tipos de Transacción")

### Listado de Variables

In [5]:
variables = pd.read_excel(os.path.join(path, "Archivos Auxiliares", "Listados Auxiliares.xlsx"), sheet_name="Descripción de Variables")

### Contenido

In [6]:
contenido = pd.read_excel(os.path.join(path, "Resultados", "OCR", "Resultados_OCR.xlsx"))
contenido["conservador_id"] = contenido["archivo"].str.extract(r'^([A-ZÁÉÍÓÚÑ ]+?)\s(?=\d)')
contenido["año_id"] = contenido["archivo"].str.extract(r'(\d{4})').astype(int)
contenido.loc[~contenido["año_id"].between(1900, 2025), "año_id"] = np.nan
contenido["id"] = contenido.apply(lambda r: r["archivo"].replace(r["conservador"], "").strip() if pd.isna(r["año_id"]) else r["archivo"].split(str(int(r["año_id"])), 1)[1].strip(), axis=1)

## Modelo de datos

In [7]:
path_modelo = (os.path.join(path, "Códigos", "Extracción", "Modelo_Datos.ipynb"))
%run "{path_modelo}"

## Instrucciones generales AI

In [8]:
path_instrucciones = (os.path.join(path, "Códigos", "Extracción", "Instrucciones.ipynb"))
%run "{path_instrucciones}"

## Funciones Modelo IA

In [9]:
path_gpt = (os.path.join(path, "Códigos", "Extracción", "Ejecución.ipynb"))
%run "{path_gpt}"

## Ejecución Loop Principal

In [10]:
# Configurar logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def procesar_fase_gpt_seguro(client, texto_base, contexto, fase_config, fase_num, max_reintentos=3):
    """
    Versión segura de procesar_fase_gpt con manejo de errores y reintentos
    """
    
        
    for intento in range(max_reintentos):
        try:
            # Llamar a tu función original
            respuesta_fase = procesar_fase_gpt(
                client=client,
                texto_base=texto_base,
                contexto=contexto,
                fase_config=fase_config,
                fase_num=fase_num
            )
            return respuesta_fase
            
        except ValidationError as e:
            # Error de validación de Pydantic
            logger.error(f"ValidationError en fase {fase_num}, intento {intento+1}/{max_reintentos}")
            logger.error(f"Detalles: {e}")
            
            # Intentar extraer el valor problemático
            if hasattr(e, 'errors'):
                for error in e.errors():
                    campo = error.get('loc', ['desconocido'])[0]
                    valor_input = error.get('input', 'N/A')
                    tipo_error = error.get('type', 'N/A')
                    logger.error(f"  Campo: {campo}, Valor: {valor_input}, Tipo: {tipo_error}")
            
            if intento < max_reintentos - 1:
                logger.info(f"Reintentando fase {fase_num}...")
                continue
            else:
                logger.error(f"Fase {fase_num} falló después de {max_reintentos} intentos")
                return {}
                
        except Exception as e:
            # Cualquier otro error
            logger.error(f"Error inesperado en fase {fase_num}, intento {intento+1}/{max_reintentos}")
            logger.error(f"Tipo: {type(e).__name__}, Mensaje: {str(e)}")
            
            if intento < max_reintentos - 1:
                logger.info(f"Reintentando fase {fase_num}...")
                continue
            else:
                logger.error(f"Fase {fase_num} falló después de {max_reintentos} intentos")
                return {}
    
    return {}

In [11]:
config_fases = []

for i in range(1, 8):
    config_fases.append({
        "id": f"fase{i}",
        "numero": f"{i}",
        "instr": globals()[f"instr_fase{i}"],    
        "examples": globals()[f"examples_{i}"],    
        "model": globals()[f"Fase{i}Registro"]    
    })

In [12]:
# Procesamiento principal con manejo de errores
resultados_finales = []
documentos_con_error = []

for idx, row in tqdm(contenido[:10].iterrows(), total=len(contenido), desc="Procesando Documentos"):
    try:
        # Contexto acumulado para este documento
        cols = ["archivo", "conservador_id", "año_id", "id"]
        contexto = {**row[cols].to_dict()}
        print(f"\nARCHIVO: {row['archivo']}")
        
        # Flag para detectar si hubo errores críticos
        error_critico = False
        
        # Procesar cada fase
        for fase in config_fases[:]:
            fase_num = fase["numero"]
            print(f"FASE: {fase_num}")
            
            # Verificar si debemos omitir esta fase
            if not debe_ejecutar_fase(fase_num, contexto.get('tipo_transaccion')):
                print(f"  Omitiendo fase {fase_num} para tipo_transaccion={contexto.get('tipo_transaccion')}")
                continue
            
            # Procesar la fase con manejo de errores
            respuesta_fase = procesar_fase_gpt_seguro(
                client=client,
                texto_base=row["contenido"],
                contexto=contexto,
                fase_config=fase,
                fase_num=fase_num,
                max_reintentos=3
            )
            print(respuesta_fase)
            
            # Si la respuesta está vacía, marcar como error pero continuar
            if not respuesta_fase:
                logger.warning(f"Fase {fase_num} devolvió respuesta vacía para {row['archivo']}")
                error_critico = True
                # Continuar con las siguientes fases aunque esta falló
            else:
                # Actualizar contexto con resultados
                contexto.update(respuesta_fase)
            
            # Extraer número de derechos
            if fase_num == '2' and 'tipo_derecho' in respuesta_fase:
                contexto['numero_derechos'] = len(respuesta_fase['tipo_derecho'])
        
        # Guardar resultado final del documento
        print(contexto)
        contexto['contenido'] = row['contenido']
        
        # Marcar si el documento tuvo errores
        if error_critico:
            contexto['procesamiento_con_errores'] = True
            documentos_con_error.append(row['archivo'])
        else:
            contexto['procesamiento_con_errores'] = False
            
        resultados_finales.append(contexto)
        df_final = pd.DataFrame(resultados_finales)
        df_final.to_excel(os.path.join(path, "Resultados", "Extracción", "resultados.xlsx"), index = False)
        
    except Exception as e:
        # Error catastrófico en el documento completo
        logger.error(f"Error catastrófico procesando documento {row['archivo']}: {e}")
        documentos_con_error.append(row['archivo'])
        
        # Agregar resultado parcial con información de error
        resultado_error = {
            'archivo': row['archivo'],
            'conservador_id': row.get('conservador_id'),
            'año_id': row.get('año_id'),
            'id': row.get('id'),
            'error': str(e),
            'procesamiento_con_errores': True,
            'contenido': row['contenido']
        }
        resultados_finales.append(resultado_error)
        df_final = pd.DataFrame(resultados_finales)
        df_final.to_excel(os.path.join(path, "Resultados", "Extracción", "resultados.xlsx"), index = False)


# Reporte final
print(f"\n{'='*60}")
print(f"PROCESAMIENTO COMPLETADO")
print(f"{'='*60}")
print(f"Total de documentos procesados: {len(df_final)}")
print(f"Documentos con errores: {len(documentos_con_error)}")
if documentos_con_error:
    print(f"\nDocumentos que presentaron errores:")
    for doc in documentos_con_error:
        print(f"  - {doc}")

# Guardar lista de documentos con error
if documentos_con_error:
    with open('documentos_con_error.txt', 'w') as f:
        f.write('\n'.join(documentos_con_error))
    print(f"\nLista guardada en: documentos_con_error.txt")

Procesando Documentos:   0%|                                                                   | 0/328 [00:00<?, ?it/s]


ARCHIVO: MELIPILLA 2024 188 103 VTA
FASE: 1


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


{'tipo_transaccion': 'RENUNCIA', 'numero_cbr_actual': 188, 'fojas_cbr_actual': '103v', 'dia_cbr_actual': 15, 'mes_cbr_actual': 5, 'ano_cbr_actual': 2024}
FASE: 2


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...


ARCHIVO: PUENTE ALTO 2025 182 113
FASE: 1


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


{'tipo_transaccion': 'HERENCIA', 'numero_cbr_actual': 182, 'fojas_cbr_actual': '113', 'dia_cbr_actual': 27, 'mes_cbr_actual': 5, 'ano_cbr_actual': 2025}
FASE: 2


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


{'tipo_derecho': ['NO INDICA/OTRO'], 'naturaleza_agua': ['NO INDICA/OTRO'], 'ejercicio_derecho': ['NO INDICA/OTRO'], 'uso_agua': ['AGRICULTURA']}
FASE: 3


## Guardar Resultados

In [13]:
df_final = pd.DataFrame(resultados_finales)

In [14]:
df_final.to_excel(os.path.join(path, "Resultados", "Extracción", "resultados.xlsx"), index = False)